# Data Cleaning

In this notebook, we show how to load and save in a more efficient way the original fi-2010 dataset. In order to easily load the data we exploit part of the code of the project [*lob-deep-learning*](https://github.com/Jeonghwan-Cheon/lob-deep-learning/tree/main), by Jeonghwan-Cheon. The loader/parser is available in the file `fi2010_loader.py`.

In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# FI-2010 Data Loader/Parser
import fi2010_loader as loader

Define which days, stocks and horizons to load, we consider every possible value for each of these parameters.

In [ ]:
days = range(1,10+1)
stocks = {
    0: 'KESBV',
    1: 'OUT1V',
    2: 'SAMPO',
    3: 'RTRKS',
    4: 'WRT1V'
}
horizons = {
    0:10,
    1:20,
    2:30,
    3:50,
    4:100
}

In [ ]:
datasets = dict()

def process_stock(stock_idx):
    # For a fixed stock, we build the dataset for all days
    for day in tqdm(days):
        curr_x = None
        curr_y = None
        for horizon in horizons.keys():
            dataset = loader.Dataset_fi2010(
                auction=False,
                normalization="Zscore",
                stock_idx=[stock_idx],
                days=[day],
                T=1,
                k=horizon,
                lighten=False
            )
            if curr_x is None:
                curr_x = dataset.x.squeeze().detach().numpy()
                curr_y = dataset.y.squeeze().detach().numpy().reshape(-1,1)
            else:
                # x is always the same, so we can just concatenate y
                y = dataset.y.squeeze().detach().numpy().reshape(-1,1)
                curr_y = np.concatenate((curr_y, y), axis=1)
        # Features names
        x_cols = [
            [ f'P_Ask_{i}', f'V_Ask_{i}', f'P_Bid_{i}', f'V_Bid_{i}' ] for i in range(10)
        ]
        x_cols = [item for sublist in x_cols for item in sublist]
        # Labels names
        y_cols = [ f'Label_{horizons[i]}' for i in horizons.keys() ]
        # Concatenate all columns
        colnames = x_cols + y_cols
        # Create a temporary dataframe
        tmp_df = pd.DataFrame(np.concatenate((curr_x, curr_y), axis=1), columns=colnames)
        tmp_df['Day'] = day
        datasets[day] = tmp_df
    # Concat all days
    df = pd.concat(datasets.values())
    # Save to parquet
    return df

We then process each stock and save the dataset in the parquet format. This allows us to store and read the data in a more efficient way.

In [ ]:
# Process all stocks
for stock_idx, stock_name in stocks.items():
    print(f'Processing stock {stock_name}')
    df = process_stock(stock_idx)
    df.to_parquet(f'./FI-2010_{stock_name}.parquet', index=False, compression='lz4')